
## 00_config — RCM Intelligence Platform
### Purpose
Single source of truth for all configuration across the entire pipeline.
Every notebook starts with: `%run ../00_setup/00_config`

### Inputs
None — this is the root config

### Outputs
All variables available to any notebook that runs this

| Field | Value |
|---|---|
| Author | Mayank Joshi |
| Layer | Setup |
| Domain | Healthcare — Revenue Cycle Management |
| Catalog | rcm_platform |

In [0]:
# ============================================================
# CATALOG & SCHEMA DEFINITIONS
# Three-level namespace: catalog.schema.table
# ============================================================

CATALOG          = "rcm_platform"

SCHEMA_BRONZE    = "rcm_bronze"
SCHEMA_SILVER    = "rcm_silver"
SCHEMA_GOLD      = "rcm_gold"
SCHEMA_ML        = "rcm_ml"
SCHEMA_AUDIT     = "rcm_audit"

BRONZE_DB        = f"{CATALOG}.{SCHEMA_BRONZE}"
SILVER_DB        = f"{CATALOG}.{SCHEMA_SILVER}"
GOLD_DB          = f"{CATALOG}.{SCHEMA_GOLD}"
ML_DB            = f"{CATALOG}.{SCHEMA_ML}"
AUDIT_DB         = f"{CATALOG}.{SCHEMA_AUDIT}"

print(f"{'Catalog':<12}: {CATALOG}")
print(f"{'Bronze':<12}: {BRONZE_DB}")
print(f"{'Silver':<12}: {SILVER_DB}")
print(f"{'Gold':<12}: {GOLD_DB}")
print(f"{'ML':<12}: {ML_DB}")
print(f"{'Audit':<12}: {AUDIT_DB}")

In [0]:
# ============================================================
# BRONZE TABLES  (raw, append-only, full audit trail)
# ============================================================

TBL_BRONZE_INPATIENT_RAW   = f"{BRONZE_DB}.inpatient_claims_raw"
TBL_BRONZE_OUTPATIENT_RAW  = f"{BRONZE_DB}.outpatient_claims_raw"
TBL_BRONZE_HOSPITAL_RAW    = f"{BRONZE_DB}.hospital_info_raw"
TBL_BRONZE_QUARANTINE      = f"{BRONZE_DB}.dq_quarantine"

print("Bronze tables:")
for t in [
    TBL_BRONZE_INPATIENT_RAW,
    TBL_BRONZE_OUTPATIENT_RAW,
    TBL_BRONZE_HOSPITAL_RAW,
    TBL_BRONZE_QUARANTINE
]:
    print(f"  {t}")

In [0]:
# ============================================================
# SILVER TABLES  (cleansed, enriched, SCD dimensions)
# ============================================================

TBL_SILVER_CLAIMS          = f"{SILVER_DB}.inpatient_claims_enriched"

# SCD Dimensions
TBL_DIM_HOSPITAL           = f"{SILVER_DB}.dim_hospital"   # SCD Type 2
TBL_DIM_PROVIDER           = f"{SILVER_DB}.dim_provider"   # SCD Type 6
TBL_DIM_PAYER              = f"{SILVER_DB}.dim_payer"      # SCD Type 2
TBL_DIM_DRG                = f"{SILVER_DB}.dim_drg"        # SCD Type 1
TBL_DIM_DATE               = f"{SILVER_DB}.dim_date"       # Static

print("Silver tables:")
for t in [
    TBL_SILVER_CLAIMS,
    TBL_DIM_HOSPITAL,
    TBL_DIM_PROVIDER,
    TBL_DIM_PAYER,
    TBL_DIM_DRG,
    TBL_DIM_DATE
]:
    print(f"  {t}")

In [0]:
# ============================================================
# GOLD TABLES  (star schema, analytics-ready)
# ============================================================

TBL_FACT_CLAIMS            = f"{GOLD_DB}.fact_claims"
TBL_GOLD_KPI_STATE         = f"{GOLD_DB}.kpi_by_state"
TBL_GOLD_KPI_DRG           = f"{GOLD_DB}.kpi_by_drg"
TBL_GOLD_SCORECARD         = f"{GOLD_DB}.hospital_scorecard"
TBL_GOLD_AR_AGING          = f"{GOLD_DB}.ar_aging_buckets"
TBL_GOLD_DENIAL_SCORES     = f"{GOLD_DB}.denial_risk_scores"

print("Gold tables:")
for t in [
    TBL_FACT_CLAIMS,
    TBL_GOLD_KPI_STATE,
    TBL_GOLD_KPI_DRG,
    TBL_GOLD_SCORECARD,
    TBL_GOLD_AR_AGING,
    TBL_GOLD_DENIAL_SCORES
]:
    print(f"  {t}")

In [0]:
# ============================================================
# AUDIT & PIPELINE METADATA
# ============================================================

TBL_AUDIT_PIPELINE_LOG     = f"{AUDIT_DB}.pipeline_run_log"

PIPELINE_NAME              = "rcm_intelligence_platform"
PIPELINE_VERSION           = "1.0.0"
PIPELINE_ENV               = "dev"          # dev | staging | prod

print(f"{'Audit log':<16}: {TBL_AUDIT_PIPELINE_LOG}")
print(f"{'Pipeline':<16}: {PIPELINE_NAME} v{PIPELINE_VERSION} [{PIPELINE_ENV}]")

In [0]:
# ============================================================
# DATA SOURCE CONFIGURATION
# CMS Open Data — no API key required
# ============================================================

CMS_INPATIENT_API  = (
    "https://data.cms.gov/api/1/datastore/query/"
    "3b96d5b5-8fa7-47b0-9c21-3a2bfa1eb3a9/0"
)

CMS_OUTPATIENT_API = (
    "https://data.cms.gov/api/1/datastore/query/"
    "fc9b00f9-5e96-40b7-8dce-09bc9dffc4a9/0"
)

CMS_HOSPITAL_INFO_CSV = (
    "https://data.cms.gov/provider-data/sites/default/files/resources/"
    "092256becd267d9eeccf73bf7d16c46b_1706746893/Hospital_General_Information.csv"
)

API_PAGE_LIMIT     = 1000
API_TIMEOUT_SECS   = 30

print("Data sources:")
print(f"  Inpatient  : {CMS_INPATIENT_API[:65]}...")
print(f"  Outpatient : {CMS_OUTPATIENT_API[:65]}...")
print(f"  Hospital   : {CMS_HOSPITAL_INFO_CSV[:65]}...")

In [0]:
# ============================================================
# DATA QUALITY THRESHOLDS
# Change these here — they flow into every DQ check automatically
# ============================================================

DQ_MIN_CHARGE_AMOUNT   = 0.01
DQ_MAX_CHARGE_AMOUNT   = 10_000_000
DQ_MIN_PAYMENT_AMOUNT  = 0.01
DQ_MIN_DISCHARGES      = 1
DQ_MAX_CTP_RATIO       = 100.0

DQ_NOT_NULL_COLS = [
    "provider_id",
    "drg_code",
    "avg_submitted_charge",
    "avg_total_payment",
    "provider_state"
]

print("DQ thresholds:")
print(f"  Charge range  : ${DQ_MIN_CHARGE_AMOUNT} — ${DQ_MAX_CHARGE_AMOUNT:,}")
print(f"  Payment min   : ${DQ_MIN_PAYMENT_AMOUNT}")
print(f"  Max CTP ratio : {DQ_MAX_CTP_RATIO}")
print(f"  Not-null cols : {DQ_NOT_NULL_COLS}")

In [0]:
# ============================================================
# FILE PATHS  (DBFS — Databricks File System)
# ============================================================

BASE_PATH            = "/FileStore/rcm_platform"
RAW_PATH             = f"{BASE_PATH}/raw"
CHECKPOINT_PATH      = f"{BASE_PATH}/checkpoints"
SCHEMA_PATH          = f"{BASE_PATH}/schemas"
TEMP_PATH            = f"{BASE_PATH}/temp"

RAW_INPATIENT_PATH   = f"{RAW_PATH}/inpatient"
RAW_OUTPATIENT_PATH  = f"{RAW_PATH}/outpatient"
RAW_HOSPITAL_PATH    = f"{RAW_PATH}/hospital_info"

print("File paths:")
for label, path in [
    ("Base",        BASE_PATH),
    ("Raw",         RAW_PATH),
    ("Checkpoints", CHECKPOINT_PATH),
    ("Schemas",     SCHEMA_PATH),
    ("Temp",        TEMP_PATH),
]:
    print(f"  {label:<14}: {path}")

In [0]:
# ============================================================
# SCD CONSTANTS
# Used by all dimension notebooks
# ============================================================

SCD_END_OF_TIME    = "9999-12-31"   # expiry date for active rows
SCD_INITIAL_DATE   = "2024-01-01"   # effective date for first load

print(f"SCD end of time  : {SCD_END_OF_TIME}")
print(f"SCD initial date : {SCD_INITIAL_DATE}")

In [0]:
# ============================================================
# CONFIG LOAD CONFIRMATION
# This prints every time a notebook successfully imports config
# ============================================================

print("=" * 60)
print("  RCM INTELLIGENCE PLATFORM — CONFIG LOADED")
print(f"  Pipeline : {PIPELINE_NAME} v{PIPELINE_VERSION} [{PIPELINE_ENV}]")
print(f"  Catalog  : {CATALOG}")
print(f"  Schemas  : {SCHEMA_BRONZE} | {SCHEMA_SILVER} | {SCHEMA_GOLD} | {SCHEMA_ML}")
print(f"  Audit    : {TBL_AUDIT_PIPELINE_LOG}")
print("=" * 60)
